# SR-EV Simulation Pipeline Demo

Demonstrates `hmt_v3.srev.pipeline` end-to-end: takes an already-generated SR-EV `config-N.txt`, splits monomers into two compartments (heterochromatin / euchromatin, via `srev.compartments`), and runs each compartment as its own independent **channel** through primary/secondary antibody labeling, Markov blinking photophysics, PSF + camera-noise rendering, and ground-truth bookkeeping -- producing two TIFF stacks (one per channel) plus two saved ground-truth run directories, ready for ThunderSTORM.

This mirrors `hmt_v1`'s original two-dye setup (one dye per histone mark/compartment) but goes through the corrected `hmt_v3.srev` pipeline instead of the legacy MATLAB code.

**Scope note**: `pipeline.run_pipeline` renders exactly one channel per call, by design -- see `hmt_v3/srev/pipeline.py`'s module docstring. A two-channel acquisition is two independent calls with different `compartment_filter` values, so each channel's labeling density, dye brightness, and camera noise stay fully independent, the way two real antibody/dye channels would be.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile
from scipy.spatial import cKDTree

from hmt_v3 import srev

## Load a config and classify compartments on the FULL monomer cloud

Compartment classification (coordination number, `srev.compartments`) counts real neighbors within a fixed radius, so it needs to run on the FULL, uncropped monomer cloud -- doing it after cropping to a small patch would undercount monomers near the crop boundary and bias them toward euchromatin. Crop for speed only AFTER classifying (next cell).

In [ ]:
CONFIG_PATH = Path("hmt_v1/Simulation/config-1.txt")  # point this at any config-N.txt(.gz)

# Prefer resolve_srev_run_params (copy-parameters.txt / folder name) when available; it returns
# whatever it can find without raising just because some fields are missing (e.g. config-1.txt's
# parent folder isn't an "SRRW-*" name and has no sibling copy-parameters.txt), so fall back to
# length_unit_to_nm=10 (Umin) -- correct for Cangnano et al. 2024 SR-EV runs -- via .get() rather
# than a try/except (a real ValueError here, from disagreeing sources, should still surface).
params = srev.io.resolve_srev_run_params(CONFIG_PATH)
length_unit_to_nm = params.get("length_unit_to_nm", 10.0)
if "length_unit_to_nm" not in params:
    print(f"note: could not resolve length_unit_to_nm from '{CONFIG_PATH.parent.name}' "
          f"(no matching SRRW-* folder name or copy-parameters.txt) -- defaulting to 10.0 nm")

monomers_full, meta = srev.io.parse_config_txt(CONFIG_PATH, length_unit_to_nm=length_unit_to_nm)
monomers_full = srev.compartments.classify_compartments(monomers_full)

print(f"{len(monomers_full)} monomers parsed from {CONFIG_PATH}")
print(monomers_full["compartment"].value_counts().to_dict())

## Crop to a fast-to-render patch

SR-EV configs can have 10^5-10^6 monomers -- far more than fit in a typical camera FOV or are worth simulating for a demo run. This takes the `N_PATCH` monomers nearest the cloud's centroid (robust to whatever physical scale this particular config turns out to have) and keeps the `compartment` column computed above intact.

In [ ]:
N_PATCH = 6000

centroid = monomers_full[["x [nm]", "y [nm]", "z [nm]"]].to_numpy().mean(axis=0)
tree = cKDTree(monomers_full[["x [nm]", "y [nm]", "z [nm]"]].to_numpy())
_, idx = tree.query(centroid, k=min(N_PATCH, len(monomers_full)))
patch_df = monomers_full.iloc[idx].reset_index(drop=True).copy()

print(f"patch: {len(patch_df)} monomers")
print(patch_df["compartment"].value_counts().to_dict())

## Configure the two channels

Each `SrEvSimConfig` nests every stage's own config (`LabelingConfig`, `PhotophysicsConfig`, `CameraConfig`, `RenderConfig`) so a parameter is declared once and used everywhere it's read -- no separately-drifting copies. The two channels below differ in labeling density and dye brightness (standing in for two different antibody/dye pairs, e.g. an anti-H3K27me3/Alexa647 heterochromatin channel vs. an anti-H3K27ac/CF680 euchromatin channel) and use different RNG seeds so they don't share randomness.

In [ ]:
config_heterochromatin = srev.pipeline.SrEvSimConfig(
    labeling=srev.labeling.LabelingConfig(labeling_efficiency=0.3, n_dye_per_secondary=("poisson", 2.0)),
    photophysics=srev.photophysics.PhotophysicsConfig(n_frames=500, mean_photons_per_frame=300.0),
    camera=srev.render.CameraConfig(pixel_size_nm=100.0),
    compartment_filter="heterochromatin",
    seed=1,
)

config_euchromatin = srev.pipeline.SrEvSimConfig(
    labeling=srev.labeling.LabelingConfig(labeling_efficiency=0.3, n_dye_per_secondary=("poisson", 2.0)),
    photophysics=srev.photophysics.PhotophysicsConfig(n_frames=500, mean_photons_per_frame=450.0),
    camera=srev.render.CameraConfig(pixel_size_nm=100.0),
    compartment_filter="euchromatin",
    seed=2,
)

## Run both channels

Each call to `run_pipeline` is fully self-contained: it filters to the requested compartment, runs labeling -> photophysics, auto-sizes the camera frame to fit the labeled emitters (`render.fit_frame_to_points`), renders + noises the movie, writes a BigTIFF stack, and saves the ground-truth tables (Parquet + `run_config.json`) -- checking referential integrity before writing anything.

In [ ]:
OUTPUT_ROOT = Path("sr_ev_pipeline_demo_output")

result_het = srev.pipeline.run_pipeline(
    patch_df, config_heterochromatin, OUTPUT_ROOT / "heterochromatin",
    source_path=CONFIG_PATH, input_meta=meta,
)
result_eu = srev.pipeline.run_pipeline(
    patch_df, config_euchromatin, OUTPUT_ROOT / "euchromatin",
    source_path=CONFIG_PATH, input_meta=meta,
)

for name, result in (("heterochromatin", result_het), ("euchromatin", result_eu)):
    print(f"--- {name} ---")
    print(f"  monomers used:      {len(result['monomers'])}")
    print(f"  primaries:          {len(result['primaries'])}")
    print(f"  secondaries:        {len(result['secondaries'])}")
    print(f"  fluorophores:       {len(result['fluorophores'])}")
    print(f"  blink events:       {len(result['blink_events'])}")
    print(f"  frame_shape (px):   {result['frame_shape']}")
    print(f"  TIFF stack:         {result['written']['tiff']}")

## Visualize both channels

A max-intensity projection across the movie for each channel, side by side -- a quick sanity check that emitters actually landed inside the frame and produced signal above the camera baseline, before handing the stacks to ThunderSTORM.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (name, result) in zip(axes, (("heterochromatin", result_het), ("euchromatin", result_eu))):
    stack = tifffile.imread(result["written"]["tiff"])
    mip = stack.max(axis=0)
    im = ax.imshow(mip, cmap="inferno")
    ax.set_title(f"{name}\n{stack.shape[0]} frames, {stack.shape[1]}x{stack.shape[2]} px")
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="ADU (max over stack)")
fig.tight_layout()
plt.show()

## Inspect the ground truth

`resolved_localizations_gt` is the direct comparison target for a real ThunderSTORM run on these stacks: one row per blink event, with the emitter's TRUE static position joined in (not what any fitter would recover).

In [ ]:
loaded_het = srev.ground_truth.load_run(OUTPUT_ROOT / "heterochromatin")
loaded_eu = srev.ground_truth.load_run(OUTPUT_ROOT / "euchromatin")

print("heterochromatin resolved_localizations_gt:")
display(loaded_het["resolved_localizations_gt"].head())

print("euchromatin resolved_localizations_gt:")
display(loaded_eu["resolved_localizations_gt"].head())

## Next step (manual, outside this notebook)

Open each channel's `stack.tif` in ImageJ/Fiji, run ThunderSTORM's real detection + fitting, and nearest-neighbor-match its output against the corresponding `resolved_localizations_gt` table in `(x, y, frame)` (e.g. via `scipy.spatial.cKDTree`) to get a first detection-rate / localization-error number. This is the actual acceptance test for the whole pipeline's purpose -- everything above only prepares the inputs for it.